# 🐍 Day 21 · Python Basics — Hand-offs & State Machines (Handoffs and Swarm)

> **Goal of this notebook:** In a hand-off system the *current* agent decides who talks next. That is a **state machine**. Build one with a `while` loop.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

✅ No API key and no internet needed for this notebook.

### 📋 What you will learn
1. `Enum` — a fixed set of names
2. Agents that return *who is next*
3. A `while` loop that follows the hand-offs
4. Tracking the path
5. Protection against endless loops

## 1. `Enum` — a fixed set of names
An **Enum** prevents typos: you can only use the names you defined.

In [ ]:
from enum import Enum

class Agent(Enum):
    INTAKE = "intake"
    CREDIT = "credit_check"
    COMPLIANCE = "compliance"
    DECISION = "decision"

print(Agent.CREDIT)
print(Agent.CREDIT.value)
print([a.value for a in Agent])

## 2. Agents that return *who is next*
Each agent returns two things: its **message** and the **next agent** (or `None` to finish).

In [ ]:
def intake(app):
    return f"Got application from {app['name']}", Agent.CREDIT

def credit_check(app):
    if app["credit_score"] < 600:
        return "Credit score too low", Agent.DECISION
    return "Credit OK", Agent.COMPLIANCE

def compliance(app):
    ok = app["income"] * 5 >= app["loan"]
    return ("Income check passed" if ok else "Loan too large for income"), Agent.DECISION

def decision(app):
    approved = app["credit_score"] >= 600 and app["income"] * 5 >= app["loan"]
    return ("✅ APPROVED" if approved else "❌ REJECTED"), None

agents = {Agent.INTAKE: intake, Agent.CREDIT: credit_check, Agent.COMPLIANCE: compliance, Agent.DECISION: decision}

## 3. 🔗 Follow the hand-offs

In [ ]:
def run(app, start=Agent.INTAKE, max_hops=10):
    current = start
    path = []                                  # which agents spoke
    while current is not None:
        if len(path) >= max_hops:              # safety net against endless loops
            print("⚠️ Too many hand-offs, stopping")
            break
        message, nxt = agents[current](app)
        path.append(current.value)
        print(f"[{current.value:<12}] {message}")
        current = nxt
    return path

app = {"name": "Meera", "credit_score": 720, "income": 50000, "loan": 200000}
print("Path:", " → ".join(run(app)))

In [ ]:
bad_app = {"name": "Sam", "credit_score": 540, "income": 30000, "loan": 500000}
print("Path:", " → ".join(run(bad_app)))       # compliance is skipped!

## 4. Endless loop protection
What if two agents keep handing over to each other? `max_hops` stops it.

In [ ]:
def ping(app): return "ping", Agent.CREDIT
def pong(app): return "pong", Agent.INTAKE
agents_loop = {Agent.INTAKE: ping, Agent.CREDIT: pong}

agents_backup = agents
agents = agents_loop
run({}, max_hops=4)
agents = agents_backup          # restore the real agents

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Run `run` with an applicant who has a credit score of 650, income 10000 and loan 100000.
2. Add an Enum member `FRAUD = "fraud_check"`.
3. Make `credit_check` print the score it checked.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print(run({"name": "Ana", "credit_score": 650, "income": 10000, "loan": 100000}))
# Compliance fails because 10000 * 5 < 100000 -> REJECTED

class Agent2(Enum):
    INTAKE = "intake"
    FRAUD = "fraud_check"
print(Agent2.FRAUD.value)
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Enum | `class Agent(Enum): CREDIT = 'credit'` |
| Enum value | `Agent.CREDIT.value` |
| Return next agent | `return message, Agent.DECISION` |
| Follow hand-offs | `while current is not None:` |
| Track path | `path.append(current.value)` |
| Safety limit | `if len(path) >= max_hops: break` |

## ➡️ What's next?
You now know the Python used in **Day_21_Multi_Agent_Handoffs_and_Swarm**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀